In [ ]:
import os
os.environ["CUDA_VISIBLE_DEVICES"] = "1,2,3,5,6,7"
import textattack
from transformers import AutoModelForSequenceClassification, AutoTokenizer
import torch
import pandas as pd
from langdetect import detect

In [1]:
from tqdm import tqdm, trange

from datasets import load_dataset
import pandas as pd
import functools
import sys
from pathlib import Path
from typing import Callable

# import circuitsvis as cv
import einops
import numpy as np
import torch as t
import torch.nn as nn
import torch.nn.functional as F
import eindex
# from IPython.display import display
from jaxtyping import Float, Int
from torch import Tensor
from tqdm import tqdm
from transformer_lens import (
    ActivationCache,
    FactoredMatrix,
    HookedTransformer,
    HookedTransformerConfig,
    HookedEncoderDecoder,
    HookedEncoder,
    utils,
)
from transformer_lens.hook_points import HookPoint

import torch
import torch.nn as nn
import torch.optim as optim
from torch.utils.data import DataLoader, Dataset
from datasets import load_dataset
from transformers import AutoTokenizer
from transformer_lens import HookedTransformer
import os
import json
import matplotlib.pyplot as plt
import seaborn as sns

/home/kokil/shaz/interp-toxicity/interp-toxicity/lib/python3.10/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
2025-06-11 13:53:18.930231: I tensorflow/core/util/port.cc:153] oneDNN custom operations are on. You may see slightly different numerical results due to floating-point round-off errors from different computation orders. To turn them off, set the environment variable `TF_ENABLE_ONEDNN_OPTS=0`.
2025-06-11 13:53:18.944652: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:467] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
E0000 00:00:1749649998.959083 1750879 cuda_dnn.cc:8579] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
E0000 00:00:1749649998.963665 1750879 

In [3]:
toxigen = load_dataset('toxigen/toxigen-data','train')['train']

In [4]:
toxigen_df = pd.DataFrame(toxigen)

In [7]:
toxigen_df.drop_duplicates(subset=['generation']).reset_index(drop=True)

,prompt,generation,generation_method,group,prompt_label,roberta_prediction
0,- the vast majority of african-americans are p...,"if youre white, you should be proud to be whit...",ALICE,black,1,0.001
1,- black people from the middle ages have alway...,the best thing that could have happened to the...,ALICE,black,1,0.044
2,- many african americans are so savage they lo...,theres no hope for the future because theres n...,ALICE,black,1,0.014
3,- the vast majority of african-americans are p...,"if youre white, you should be proud of it.",ALICE,black,1,0.002
4,"- whether a scientist or janitor, black people...","no matter how much money or power you have, if...",ALICE,black,1,0.328
...,...,...,...,...,...,...
246049,- although traditionally women were not allowe...,My Chinese name is ai-lun (�媈礍) .\n- My famil...,top-k,women,0,0.006
246050,- although traditionally women were not allowe...,I can't believe they won,top-k,women,0,0.720
246051,- although traditionally women were not allowe...,fín had a reputation as a musicians' city,top-k,women,0,0.001
246052,- although traditionally women were not allowe...,I am thinking about getting a pet dog,top-k,women,0,0.006


In [8]:
train_df = toxigen_df.groupby('prompt_label').sample(frac=0.75)
test_df = toxigen_df.drop(train_df.index).reset_index(drop=True)
train_df.reset_index(drop=True, inplace=True)

In [9]:
train_df.to_csv('toxigen/train.csv', index=False)
test_df.to_csv('toxigen/test.csv', index=False)

In [10]:
toxigen_df.generation.apply(len).describe()

count    250951.000000
mean         88.983403
std          41.533473
min           1.000000
25%          54.000000
50%          88.000000
75%         124.000000
max        1736.000000
Name: generation, dtype: float64